<a href="https://colab.research.google.com/github/rahulbalaji13/SAFERL_Hybrid_RL_FW/blob/main/pipeline_yolo_ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics stable-baselines3 gymnasium opencv-python numpy pandas torch torchvision matplotlib pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.4/965.4 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
"""
Complete YOLO + PPO Pipeline for Construction Site PPE Safety Monitoring
Single-file implementation combining YOLO object detection with PPO reinforcement learning
"""

import os
import cv2
import numpy as np
import torch
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from ultralytics import YOLO
import yaml
import shutil
import tempfile
from pathlib import Path
import logging
import json
import random
from typing import Dict, List, Tuple, Optional, Any
import time

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lbquctrung/worksite-safety-monitoring-dataset")

print("Path to dataset files:", path)

100%|██████████| 98.9M/98.9M [00:03<00:00, 34.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/lbquctrung/worksite-safety-monitoring-dataset/versions/1


In [4]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class PPEDatasetManager:
    """Manages PPE dataset creation and preparation for YOLO training"""

    def __init__(self, base_dir: str = "/root/.cache/kagglehub/datasets/lbquctrung/worksite-safety-monitoring-dataset/versions/1"):
        self.base_dir = Path(base_dir)
        self.classes = {
            0: "person",
            1: "helmet",
            2: "no_helmet",
            3: "vest",
            4: "no_vest",
            5: "mask",
            6: "no_mask"
        }

    def create_sample_dataset(self, num_samples: int = 1000) -> None:
        """Create sample dataset structure for YOLO training"""
        logger.info("Creating sample dataset structure...")

        # Create directory structure
        for split in ['train', 'val', 'test']:
            (self.base_dir / split / 'images').mkdir(parents=True, exist_ok=True)
            (self.base_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

        # Create data.yaml for YOLO
        data_yaml = {
            'path': str(self.base_dir),
            'train': 'train/images',
            'val': 'val/images',
            'test': 'test/images',
            'nc': len(self.classes),
            'names': list(self.classes.values())
        }

        with open(self.base_dir / 'data.yaml', 'w') as f:
            yaml.dump(data_yaml, f)

        # Generate synthetic dataset (placeholder - replace with your actual dataset)
        self._generate_synthetic_data(num_samples)

    def _generate_synthetic_data(self, num_samples: int) -> None:
        """Generate synthetic training data (replace with your actual data loading)"""
        logger.info("Generating synthetic training data...")

        splits = {'train': 0.7, 'val': 0.2, 'test': 0.1}

        for split, ratio in splits.items():
            split_samples = int(num_samples * ratio)

            for i in range(split_samples):
                # Create dummy image (640x640)
                img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
                img_path = self.base_dir / split / 'images' / f'image_{i:04d}.jpg'
                cv2.imwrite(str(img_path), img)

                # Create dummy label
                label_path = self.base_dir / split / 'labels' / f'image_{i:04d}.txt'
                with open(label_path, 'w') as f:
                    # Random bounding boxes for different classes
                    num_objects = random.randint(1, 5)
                    for _ in range(num_objects):
                        class_id = random.randint(0, len(self.classes) - 1)
                        x_center = random.uniform(0.1, 0.9)
                        y_center = random.uniform(0.1, 0.9)
                        width = random.uniform(0.05, 0.3)
                        height = random.uniform(0.05, 0.3)
                        f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

In [5]:
class YOLOPPEDetector:
    """YOLO-based PPE detector with training capabilities"""

    def __init__(self, model_path: Optional[str] = None):
        self.model = None
        self.model_path = model_path
        self.classes = {
            0: "person", 1: "helmet", 2: "no_helmet", 3: "vest",
            4: "no_vest", 5: "mask", 6: "no_mask"
        }

    def train_model(self, dataset_path: str, epochs: int = 50) -> None:
        """Train YOLO model from scratch"""
        logger.info("Starting YOLO training from scratch...")

        # Initialize YOLO model
        self.model = YOLO('yolov8n.yaml')  # Create new model from scratch

        # Train the model
        results = self.model.train(
            data=f"{dataset_path}/data.yaml",
            epochs=epochs,
            imgsz=640,
            batch=16,
            device='cpu',  # Use 'cuda' if GPU available
            workers=4,
            optimizer='SGD',
            lr0=0.01,
            patience=10,
            save_period=10,
            cache=False,
            amp=False,  # Disable mixed precision for CPU
            project='runs/train',
            name='ppe_detector'
        )

        # Save the best model
        self.model_path = results.save_dir / 'weights' / 'best.pt'
        logger.info(f"Model saved to: {self.model_path}")

    def load_model(self, model_path: str) -> None:
        """Load trained YOLO model"""
        self.model = YOLO(model_path)
        self.model_path = model_path

    def detect_ppe(self, image: np.ndarray) -> Dict[str, Any]:
        """Detect PPE in image and return structured results"""
        if self.model is None:
            raise ValueError("Model not loaded. Call train_model() or load_model() first.")

        results = self.model(image, conf=0.3, iou=0.5, verbose=False)

        detections = {
            'person_count': 0,
            'helmet_count': 0,
            'no_helmet_count': 0,
            'vest_count': 0,
            'no_vest_count': 0,
            'mask_count': 0,
            'no_mask_count': 0,
            'total_violations': 0,
            'safety_score': 0.0,
            'avg_confidence': 0.0,
            'boxes': [],
            'confidences': []
        }

        if results and len(results) > 0:
            boxes = results[0].boxes
            if boxes is not None:
                for box in boxes:
                    cls = int(box.cls.cpu().numpy()[0])
                    conf = float(box.conf.cpu().numpy()[0])

                    class_name = self.classes[cls]
                    detections[f'{class_name}_count'] += 1
                    detections['confidences'].append(conf)

                    # Count violations
                    if class_name in ['no_helmet', 'no_vest', 'no_mask']:
                        detections['total_violations'] += 1

                # Calculate safety metrics
                total_detections = len(boxes)
                if total_detections > 0:
                    detections['safety_score'] = 1.0 - (detections['total_violations'] / total_detections)
                    detections['avg_confidence'] = np.mean(detections['confidences'])
                else:
                    detections['safety_score'] = 1.0
                    detections['avg_confidence'] = 0.0

        return detections

In [6]:
class PPEMonitoringEnvironment(gym.Env):
    """Custom Gym environment for PPE monitoring with PPO"""

    def __init__(self, yolo_detector: YOLOPPEDetector, config: Dict[str, Any]):
        super().__init__()

        self.yolo_detector = yolo_detector
        self.config = config

        # Action space: 0=No Action, 1=Raise Alert, 2=Request Human Review
        self.action_space = spaces.Discrete(3)

        # Observation space: safety metrics from YOLO
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(10,), dtype=np.float32
        )

        # Environment state
        self.current_image = None
        self.episode_length = 0
        self.max_episode_length = config.get('max_episode_length', 100)
        self.alert_threshold = config.get('alert_threshold', 0.5)
        self.human_review_threshold = config.get('human_review_threshold', 0.3)

        # Metrics
        self.total_alerts = 0
        self.correct_alerts = 0
        self.false_alarms = 0
        self.missed_violations = 0

    def reset(self, seed: Optional[int] = None) -> Tuple[np.ndarray, Dict]:
        """Reset environment"""
        super().reset(seed=seed)

        self.episode_length = 0
        self.current_image = self._generate_sample_image()

        observation = self._get_observation()
        info = {}

        return observation, info

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        """Execute one step in the environment"""
        self.episode_length += 1

        # Get current detections
        detections = self.yolo_detector.detect_ppe(self.current_image)

        # Calculate reward based on action and current state
        reward = self._calculate_reward(action, detections)

        # Generate next observation
        self.current_image = self._generate_sample_image()
        observation = self._get_observation()

        # Check if episode is done
        done = self.episode_length >= self.max_episode_length
        truncated = False

        # Update metrics
        self._update_metrics(action, detections)

        info = {
            'detections': detections,
            'action_taken': action,
            'safety_score': detections['safety_score'],
            'total_alerts': self.total_alerts,
            'correct_alerts': self.correct_alerts,
            'false_alarms': self.false_alarms
        }

        return observation, reward, done, truncated, info

    def _get_observation(self) -> np.ndarray:
        """Get current observation from YOLO detections"""
        detections = self.yolo_detector.detect_ppe(self.current_image)

        # Normalize observation features
        obs = np.array([
            detections['person_count'] / 10.0,  # Max 10 people
            detections['helmet_count'] / 10.0,
            detections['no_helmet_count'] / 10.0,
            detections['vest_count'] / 10.0,
            detections['no_vest_count'] / 10.0,
            detections['mask_count'] / 10.0,
            detections['no_mask_count'] / 10.0,
            detections['total_violations'] / 10.0,
            detections['safety_score'],
            detections['avg_confidence']
        ], dtype=np.float32)

        return obs

    def _calculate_reward(self, action: int, detections: Dict[str, Any]) -> float:
        """Calculate reward based on action and current state"""
        safety_score = detections['safety_score']
        violation_count = detections['total_violations']

        reward = 0.0

        # Action 0: No Action
        if action == 0:
            if safety_score > self.alert_threshold:
                reward = 2.0  # Correct no action
            else:
                reward = -5.0  # Missed violation
                self.missed_violations += 1

        # Action 1: Raise Alert
        elif action == 1:
            if safety_score <= self.alert_threshold:
                reward = 10.0  # Correct alert
                self.correct_alerts += 1
            else:
                reward = -3.0  # False alarm
                self.false_alarms += 1
            self.total_alerts += 1

        # Action 2: Request Human Review
        elif action == 2:
            if self.human_review_threshold <= safety_score <= self.alert_threshold:
                reward = 5.0  # Appropriate human review
            else:
                reward = -1.0  # Unnecessary human review

        # Bonus for high confidence detections
        if detections['avg_confidence'] > 0.8:
            reward += 1.0

        return reward

    def _generate_sample_image(self) -> np.ndarray:
        """Generate sample image (replace with actual image stream)"""
        # For demo purposes, generate random image
        # In real implementation, this would come from camera feed
        img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

        # Add some structure to make it more realistic
        cv2.rectangle(img, (100, 100), (300, 400), (50, 50, 50), -1)  # Person shape
        cv2.circle(img, (200, 150), 30, (0, 255, 0), -1)  # Helmet (green)
        cv2.rectangle(img, (150, 200), (250, 300), (0, 0, 255), -1)  # Vest (red)

        return img

    def _update_metrics(self, action: int, detections: Dict[str, Any]) -> None:
        """Update environment metrics"""
        pass  # Metrics are updated in _calculate_reward

    def render(self, mode: str = 'human') -> None:
        """Render environment (optional)"""
        if mode == 'human':
            detections = self.yolo_detector.detect_ppe(self.current_image)
            print(f"Episode: {self.episode_length}, Safety Score: {detections['safety_score']:.2f}")

class PPEMonitoringSystem:
    """Main system combining YOLO detection with PPO decision making"""

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.yolo_detector = YOLOPPEDetector()
        self.ppo_agent = None
        self.environment = None

    def setup_system(self) -> None:
        """Setup the complete system"""
        logger.info("Setting up PPE monitoring system...")

        # 1. Setup dataset and train YOLO
        dataset_manager = PPEDatasetManager("ppe_dataset")
        dataset_manager.create_sample_dataset(num_samples=1000)

        # 2. Train YOLO model
        self.yolo_detector.train_model("ppe_dataset", epochs=self.config['yolo_epochs'])

        # 3. Create environment
        self.environment = PPEMonitoringEnvironment(self.yolo_detector, self.config)

        # 4. Setup PPO agent
        self.ppo_agent = PPO(
            policy="MlpPolicy",
            env=self.environment,
            learning_rate=self.config['ppo_learning_rate'],
            n_steps=self.config['ppo_n_steps'],
            batch_size=self.config['ppo_batch_size'],
            n_epochs=self.config['ppo_n_epochs'],
            gamma=self.config['ppo_gamma'],
            gae_lambda=self.config['ppo_gae_lambda'],
            clip_range=self.config['ppo_clip_range'],
            verbose=1,
            device='cpu'
        )

        logger.info("System setup complete!")

    def train_ppo_agent(self, total_timesteps: int = 10000) -> None:
        """Train the PPO agent"""
        logger.info("Training PPO agent...")

        if self.ppo_agent is None:
            raise ValueError("PPO agent not initialized. Call setup_system() first.")

        self.ppo_agent.learn(total_timesteps=total_timesteps)

        # Save trained model
        self.ppo_agent.save("ppe_ppo_agent")
        logger.info("PPO agent training complete!")

    def evaluate_system(self, num_episodes: int = 10) -> Dict[str, float]:
        """Evaluate the complete system"""
        logger.info("Evaluating system performance...")

        total_rewards = []
        total_alerts = 0
        correct_alerts = 0
        false_alarms = 0

        for episode in range(num_episodes):
            obs, _ = self.environment.reset()
            episode_reward = 0
            done = False

            while not done:
                action, _ = self.ppo_agent.predict(obs, deterministic=True)
                obs, reward, done, truncated, info = self.environment.step(action)
                episode_reward += reward

                if done or truncated:
                    break

            total_rewards.append(episode_reward)
            total_alerts += info.get('total_alerts', 0)
            correct_alerts += info.get('correct_alerts', 0)
            false_alarms += info.get('false_alarms', 0)

        metrics = {
            'avg_reward': np.mean(total_rewards),
            'std_reward': np.std(total_rewards),
            'total_alerts': total_alerts,
            'correct_alerts': correct_alerts,
            'false_alarms': false_alarms,
            'precision': correct_alerts / total_alerts if total_alerts > 0 else 0,
            'alert_rate': total_alerts / (num_episodes * self.config['max_episode_length'])
        }

        logger.info(f"Evaluation complete: {metrics}")
        return metrics

    def run_monitoring_loop(self, num_steps: int = 100) -> None:
        """Run the monitoring system in a loop"""
        logger.info("Starting monitoring loop...")

        obs, _ = self.environment.reset()

        for step in range(num_steps):
            action, _ = self.ppo_agent.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = self.environment.step(action)

            # Print action taken
            action_names = ['No Action', 'Raise Alert', 'Request Human Review']
            safety_score = info['detections']['safety_score']

            print(f"Step {step + 1}: Action={action_names[action]}, "
                  f"Safety Score={safety_score:.2f}, Reward={reward:.2f}")

            if done or truncated:
                obs, _ = self.environment.reset()

        logger.info("Monitoring loop complete!")

def main():
    """Main function to run the complete pipeline"""

    # Configuration
    config = {
        # YOLO training parameters
        'yolo_epochs': 20,  # Reduced for demo

        # PPO parameters
        'ppo_learning_rate': 3e-4,
        'ppo_n_steps': 512,
        'ppo_batch_size': 64,
        'ppo_n_epochs': 4,
        'ppo_gamma': 0.99,
        'ppo_gae_lambda': 0.95,
        'ppo_clip_range': 0.2,

        # Environment parameters
        'max_episode_length': 50,  # Reduced for demo
        'alert_threshold': 0.6,
        'human_review_threshold': 0.4
    }

    print("=" * 60)
    print("YOLO + PPO PPE Monitoring System")
    print("=" * 60)

    # Initialize system
    monitoring_system = PPEMonitoringSystem(config)

    try:
        # Setup system (train YOLO, create environment, setup PPO)
        monitoring_system.setup_system()

        # Train PPO agent
        monitoring_system.train_ppo_agent(total_timesteps=5000)  # Reduced for demo

        # Evaluate system
        evaluation_metrics = monitoring_system.evaluate_system(num_episodes=5)

        print("\nSystem Performance Metrics:")
        print(f"Average Reward: {evaluation_metrics['avg_reward']:.2f}")
        print(f"Alert Precision: {evaluation_metrics['precision']:.2f}")
        print(f"Alert Rate: {evaluation_metrics['alert_rate']:.2f}")

        # Run monitoring loop
        print("\nRunning monitoring demonstration...")
        monitoring_system.run_monitoring_loop(num_steps=20)

        print("\nSystem demonstration complete!")

    except Exception as e:
        logger.error(f"Error in system execution: {e}")
        raise

if __name__ == "__main__":
    main()


YOLO + PPO PPE Monitoring System
New https://pypi.org/project/ultralytics/8.3.168 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.167 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.00GHz)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ppe_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=ppe_detector, nbs=64, nms=

100%|██████████| 755k/755k [00:00<00:00, 36.3MB/s]

Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

YOLOv8n summary: 129 layers, 3,012,213 parameters, 3,012,197 gradients, 8.2 GFLOPs

Freezing layer 'model.22.dfl.conv.weight'
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3490.3±1299.8 MB/s, size: 469.1 KB)


train: Scanning /content/ppe_dataset/train/labels... 700 images, 0 backgrounds, 0 corrupt: 100%|██████████| 700/700 [00:00<00:00, 1959.31it/s]

...


      15/20         0G      2.721      4.685      2.886         37        640: 100%|██████████| 44/44 [06:33<00:00,  8.94s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:52<00:00,  7.56s/it]

                   all        200        633   3.81e-05     0.0266   3.54e-05   7.77e-06
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 5, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

15 epochs completed in 1.846 hours.


Optimizer stripped from runs/train/ppe_detector/weights/last.pt, 6.2MB
Optimizer stripped from runs/train/ppe_detector/weights/best.pt, 6.2MB

Validating runs/train/ppe_detector/weights/best.pt...
Ultralytics 8.3.167 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.00GHz)
YOLOv8n summary (fused): 72 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:39<00:00,  5.65s/it]


                   all        200        633    0.00179     0.0661   0.000977   0.000314
                person         84        107    0.00915      0.028    0.00476   0.000939
                helmet         70         86    0.00188     0.0116    0.00101    0.00101
             no_helmet         79         91          0          0          0          0
                  vest         65         81          0          0          0          0
               no_vest         71         86   0.000865     0.0233   0.000497   4.97e-05
                  mask         71         85   0.000613        0.4   0.000573   0.000198
               no_mask         79         97          0          0          0          0
Speed: 3.9ms preprocess, 171.9ms inference, 0.0ms loss, 7.0ms postprocess per image
Results saved to runs/train/ppe_detector
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|